# BIZ-3 | Financial Feasibility & Capital Structuring
**Contributor:** Manthan  
**Branch:** BIZ-3-financial-schemes  
**Module:** 3 — Financial Feasibility, Capital Structuring and Restructuring

---
This notebook covers deterministic financial calculations, MILP-based capital optimization and Monte Carlo scenario analysis.

In [ ]:
# BIZ-3 add financial feasibility notebook and project docs

import numpy as np
import pandas as pd

# TODO: Uncomment after environment setup
# import pulp
# from statsmodels.tsa.arima.model import ARIMA
# from sentence_transformers import SentenceTransformer

print('BIZ-3 | Financial Feasibility & Capital Structuring | Manthan')

## 1. Financial Inputs

In [ ]:
# Example entrepreneur inputs (to be collected via Module 5 conversational interface)
inputs = {
    'project_cost': 500_000,          # Total project cost (INR)
    'own_capital': 100_000,           # Entrepreneur's own savings (INR)
    'monthly_revenue_estimate': 80_000,
    'fixed_costs_monthly': 25_000,
    'variable_cost_ratio': 0.30,      # Variable costs as fraction of revenue
    'loan_interest_rate': 0.10,       # Annual interest rate
    'loan_tenure_months': 60,
    'scheme_subsidy': 50_000,         # Government subsidy (INR)
}

pd.Series(inputs).rename('Value')

## 2. Deterministic Financial Engine

In [ ]:
def calculate_emi(principal: float, annual_rate: float, months: int) -> float:
    """Standard EMI formula: P * r * (1+r)^n / ((1+r)^n - 1)"""
    r = annual_rate / 12
    return principal * r * (1 + r)**months / ((1 + r)**months - 1)

def calculate_breakeven(fixed_costs: float, variable_cost_ratio: float) -> float:
    """Break-even revenue = Fixed Costs / (1 - Variable Cost Ratio)"""
    return fixed_costs / (1 - variable_cost_ratio)

def calculate_dscr(net_operating_income: float, debt_service: float) -> float:
    """DSCR = Net Operating Income / Debt Service. Target >= 1.25"""
    return net_operating_income / debt_service if debt_service else float('inf')

def calculate_funding_gap(project_cost, own_capital, scheme_subsidy=0) -> float:
    return max(0, project_cost - own_capital - scheme_subsidy)

# Run calculations
funding_gap = calculate_funding_gap(
    inputs['project_cost'], inputs['own_capital'], inputs['scheme_subsidy']
)
emi = calculate_emi(funding_gap, inputs['loan_interest_rate'], inputs['loan_tenure_months'])
net_income = inputs['monthly_revenue_estimate'] - inputs['fixed_costs_monthly'] \
             - inputs['monthly_revenue_estimate'] * inputs['variable_cost_ratio']
cashflow = net_income - emi
dscr = calculate_dscr(net_income, emi)
breakeven = calculate_breakeven(inputs['fixed_costs_monthly'], inputs['variable_cost_ratio'])

results = {
    'Funding Gap (INR)': round(funding_gap, 2),
    'Monthly EMI (INR)': round(emi, 2),
    'Net Operating Income (INR)': round(net_income, 2),
    'Monthly Cash Flow (INR)': round(cashflow, 2),
    'DSCR': round(dscr, 3),
    'DSCR Status': 'Healthy (>=1.25)' if dscr >= 1.25 else 'At Risk (<1.25)',
    'Break-even Revenue (INR/month)': round(breakeven, 2),
}

pd.Series(results).rename('Result')

## 3. Monte Carlo Cash Flow Simulation

In [ ]:
np.random.seed(42)
N = 10_000
revenue_std = 12_000  # Assumed revenue volatility

simulated_revenues = np.random.normal(inputs['monthly_revenue_estimate'], revenue_std, N)
simulated_costs = inputs['fixed_costs_monthly'] + simulated_revenues * inputs['variable_cost_ratio']
simulated_cashflows = simulated_revenues - simulated_costs - emi

mc_summary = {
    'Mean Cash Flow': round(float(np.mean(simulated_cashflows)), 2),
    'Std Dev': round(float(np.std(simulated_cashflows)), 2),
    'P5 (Worst 5%)': round(float(np.percentile(simulated_cashflows, 5)), 2),
    'P95 (Best 5%)': round(float(np.percentile(simulated_cashflows, 95)), 2),
    'Prob. of Negative Cash Flow': f"{np.mean(simulated_cashflows < 0):.2%}",
}

pd.Series(mc_summary).rename('Monte Carlo Result')

## 4. Capital Optimization — MILP (Stub)

**Objective:** Minimize total interest + risk penalty  
**Subject to:** own_capital + loan + subsidy == project_cost, DSCR >= 1.25

In [ ]:
# TODO: Implement with PuLP
# import pulp
# prob = pulp.LpProblem('capital_structure', pulp.LpMinimize)
# ...

print('MILP optimization pending — PuLP integration in progress (BIZ-3)')

## 5. Next Steps
- [ ] Integrate PuLP for MILP capital structure optimization
- [ ] Add ARIMA/SARIMA for businesses with historical revenue records
- [ ] Expose financial engine as a FastAPI service
- [ ] Connect scheme subsidy values from Module 4 output
- [ ] Pass DSCR and cash-flow results to Module 7 explainability layer